In [1]:
%load_ext autoreload
%autoreload 2
%matplotlib inline

import os
current_pwd = os.getcwd()

possible_paths = [
    '/home/export/soheuny/SRFinder/soheun/notebooks', 
    '/home/soheuny/HH4bsim/soheun/notebooks'
]
    
assert os.getcwd() in possible_paths, f"Did you change the path? It should be one of {possible_paths}"
os.chdir("..")

In [2]:
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from plots import hist_events_by_labels
from events_data import EventsData
from fvt_classifier import FvTClassifier
# import LogNorm
from matplotlib.colors import LogNorm
from training_info import TrainingInfo
from plots import plot_rewighted_samples_by_model, plot_samples_raw
from dataset import MotherSamples
from events_data import events_from_scdinfo
import pickle

features = [
    "sym_Jet0_pt", "sym_Jet1_pt", "sym_Jet2_pt", "sym_Jet3_pt",
    "sym_Jet0_eta", "sym_Jet1_eta", "sym_Jet2_eta", "sym_Jet3_eta",
    "sym_Jet0_phi", "sym_Jet1_phi", "sym_Jet2_phi", "sym_Jet3_phi",  
    "sym_Jet0_m", "sym_Jet1_m", "sym_Jet2_m", "sym_Jet3_m",
]

# use tex
plt.rcParams["text.usetex"] = True
# plt.rcParams["font.family"] = "serif"
# plt.rcParams["font.serif"] = "Times New Roman"

plt.rcParams["figure.dpi"] = 100
plt.rcParams["figure.titlesize"] = 20
plt.rcParams["axes.titlesize"] = 20
plt.rcParams["axes.labelsize"] = 15
plt.rcParams["figure.labelsize"] = 20
plt.rcParams["lines.markersize"] = 3

import pandas as pd

# path_3b = Path("../events/MG3/dataframes/threeTag_picoAOD.h5")
# path_4b = Path("../events/MG3/dataframes/fourTag_10x_picoAOD.h5")
# path_signal = Path("../events/MG3/dataframes/HH4b_picoAOD.h5")
# df_3b = pd.read_hdf(path_3b)
# df_bg4b = pd.read_hdf(path_4b)
# df_signal = pd.read_hdf(path_signal)
# df_3b["signal"] = False
# df_bg4b["signal"] = False
# df_signal["signal"] = True
# raw_df_list = [df_3b, df_bg4b, df_signal]
# loaded_df = {path_3b: df_3b, path_4b: df_bg4b, path_signal: df_signal}

In [3]:
TrainingInfo.update_metadata()

2025-04-30 14:18:15,083 - INFO - Adding 0 files, removing 0 hashes
0it [00:00, ?it/s]


In [4]:
metadata = TrainingInfo.load_metadata()
hparams = metadata.values()
experiment_names = [hp["experiment_name"] for hp in hparams]
experiment_names = np.unique(experiment_names)
experiment_names

array(['CR_fvt_training', 'CR_fvt_training_ensemble_max',
       'CR_fvt_training_ensemble_max_HH4b_400',
       'CR_fvt_training_ensemble_max_HH4b_800',
       'CR_fvt_training_ensemble_max_fvt',
       'CR_fvt_training_ensemble_max_fvt_HH4b_400',
       'CR_fvt_training_ensemble_max_smeared',
       'CR_fvt_training_ensemble_max_smeared_HH4b_400',
       'CR_fvt_training_ensemble_mean', 'CR_fvt_training_same_data',
       'CR_fvt_training_v2', 'base_fvt_training_ensemble',
       'base_fvt_training_ensemble_HH4b_400',
       'base_fvt_training_ensemble_HH4b_800',
       'base_fvt_training_ensemble_HH4b_resonant',
       'base_fvt_training_fixed_data_split', 'better_base_fvt_training',
       'better_base_fvt_training_small', 'mi_test',
       'smeared_fvt_training', 'smeared_fvt_training_ensemble',
       'smeared_fvt_training_ensemble_HH4b_400',
       'smeared_fvt_training_ensemble_HH4b_800',
       'smeared_fvt_training_ensemble_HH4b_resonant',
       'smeared_fvt_training_noise_s

In [ ]:
recent_experiment_names = [
    # "CR_fvt_training_ensemble_max_fvt", 
    # "CR_fvt_training_ensemble_max_fvt_HH4b_400",
    # "CR_fvt_training_ensemble_max_smeared", 
    # "CR_fvt_training_ensemble_max_smeared_HH4b_400",
    "CR_fvt_training_ensemble_mean",
    "CR_fvt_training_ensemble_max",
    "CR_fvt_training_ensemble_max_HH4b_400",
    "CR_fvt_training_ensemble_max_HH4b_800",
    "base_fvt_training_ensemble",
    "base_fvt_training_ensemble_HH4b_400",
    "base_fvt_training_ensemble_HH4b_800",
    "smeared_fvt_training_ensemble",
    "smeared_fvt_training_ensemble_HH4b_400",
    "smeared_fvt_training_ensemble_HH4b_800",
]

for experiment_name in recent_experiment_names:
    hashes = TrainingInfo.find(
        {"experiment_name": experiment_name},
        return_hparams=False, use_cached_metadata=False)
    print(experiment_name, len(hashes))
    hashes_sorted = sorted(hashes)
    # print(hashes_sorted[:2])
    # print(hashes_sorted[-2:])


CR_fvt_training_ensemble_mean 5000
CR_fvt_training_ensemble_max 5000
CR_fvt_training_ensemble_max_HH4b_400 4000
CR_fvt_training_ensemble_max_HH4b_800 4000
base_fvt_training_ensemble 3750
base_fvt_training_ensemble_HH4b_400 3000
base_fvt_training_ensemble_HH4b_800 3000
smeared_fvt_training_ensemble 22500


# STEP 1 Sanity Check

In [6]:
for experiment_name in ["base_fvt_training_ensemble", 
                        "base_fvt_training_ensemble_HH4b_400", 
                        "base_fvt_training_ensemble_HH4b_800"]:
    print(experiment_name)
    sanity_list = []
    hashes, hparams = TrainingInfo.find(
        {"experiment_name": experiment_name},
        return_hparams=True,
        use_cached_metadata=False
    )
    for hp in hparams:
        sr = hp["dataset"]["signal_ratio"]
        seed = hp["dataset"]["seed"]
        train_seed = hp["train_seed"]
        model_seed = hp["model_seed"]
        data_seed = hp["data_seed"]
        assert train_seed == model_seed == data_seed
        sanity_list.append((sr, seed, train_seed))

    unique_list, counts = np.unique(sanity_list, return_counts=True, axis=0)
    print(unique_list, counts)
    assert np.all(counts == 1)
    if experiment_name == "base_fvt_training_ensemble":
        assert len(unique_list) == 5 * 50 * 15
    elif experiment_name == "base_fvt_training_ensemble_HH4b_400":
        assert len(unique_list) == 4 * 50 * 15
    elif experiment_name == "base_fvt_training_ensemble_HH4b_800":
        assert len(unique_list) == 4 * 50 * 15


base_fvt_training_ensemble
[[0.0e+00 0.0e+00 0.0e+00]
 [0.0e+00 0.0e+00 1.0e+00]
 [0.0e+00 0.0e+00 2.0e+00]
 ...
 [2.0e-02 4.9e+01 1.2e+01]
 [2.0e-02 4.9e+01 1.3e+01]
 [2.0e-02 4.9e+01 1.4e+01]] [1 1 1 ... 1 1 1]
base_fvt_training_ensemble_HH4b_400
[[5.0e-03 0.0e+00 0.0e+00]
 [5.0e-03 0.0e+00 1.0e+00]
 [5.0e-03 0.0e+00 2.0e+00]
 ...
 [2.0e-02 4.9e+01 1.2e+01]
 [2.0e-02 4.9e+01 1.3e+01]
 [2.0e-02 4.9e+01 1.4e+01]] [1 1 1 ... 1 1 1]
base_fvt_training_ensemble_HH4b_800
[[5.0e-03 0.0e+00 0.0e+00]
 [5.0e-03 0.0e+00 1.0e+00]
 [5.0e-03 0.0e+00 2.0e+00]
 ...
 [2.0e-02 4.9e+01 1.2e+01]
 [2.0e-02 4.9e+01 1.3e+01]
 [2.0e-02 4.9e+01 1.4e+01]] [1 1 1 ... 1 1 1]


# STEP 2 Sanity Check

In [7]:
for experiment_name in ["smeared_fvt_training_ensemble", 
                        "smeared_fvt_training_ensemble_HH4b_400", 
                        "smeared_fvt_training_ensemble_HH4b_800"]:
    print(experiment_name)
    sanity_list = []
    hashes, hparams = TrainingInfo.find(
        {"experiment_name": experiment_name},
        return_hparams=True,
        use_cached_metadata=False
    )
    for hp in hparams:
        sr = hp["dataset"]["signal_ratio"]
        seed = hp["dataset"]["seed"]
        train_seed = hp["train_seed"]
        model_seed = hp["model_seed"]
        data_seed = hp["data_seed"]
        noise_scale = hp["smearing"]["noise_scale"]
        assert train_seed == model_seed == data_seed
        sanity_list.append((sr, seed, train_seed, noise_scale))

    unique_list, counts = np.unique(sanity_list, return_counts=True, axis=0)
    print(unique_list, counts)
    assert np.all(counts == 1)
    if experiment_name == "smeared_fvt_training_ensemble":
        if len(unique_list) != 5 * 50 * 15 * 6:
            print(f"Total number of unique configs is not correct, should be {5 * 50 * 15 * 6}")
            print(len(unique_list))
    elif experiment_name == "smeared_fvt_training_ensemble_HH4b_400":
        if len(unique_list) != 4 * 50 * 15 * 4:
            print(f"Total number of unique configs is not correct, should be {4 * 50 * 15 * 4}")
            print(len(unique_list))
    elif experiment_name == "smeared_fvt_training_ensemble_HH4b_800":
        if len(unique_list) != 4 * 50 * 15 * 4:
            print(f"Total number of unique configs is not correct, should be {4 * 50 * 15 * 4}")
            print(len(unique_list))


smeared_fvt_training_ensemble
[[0.0e+00 0.0e+00 0.0e+00 5.0e-01]
 [0.0e+00 0.0e+00 0.0e+00 1.0e+00]
 [0.0e+00 0.0e+00 0.0e+00 1.5e+00]
 ...
 [2.0e-02 4.9e+01 1.4e+01 2.0e+00]
 [2.0e-02 4.9e+01 1.4e+01 2.5e+00]
 [2.0e-02 4.9e+01 1.4e+01 3.0e+00]] [1 1 1 ... 1 1 1]
smeared_fvt_training_ensemble_HH4b_400
[[5.0e-03 0.0e+00 0.0e+00 5.0e-01]
 [5.0e-03 0.0e+00 0.0e+00 1.0e+00]
 [5.0e-03 0.0e+00 0.0e+00 2.0e+00]
 ...
 [2.0e-02 4.9e+01 1.4e+01 1.0e+00]
 [2.0e-02 4.9e+01 1.4e+01 2.0e+00]
 [2.0e-02 4.9e+01 1.4e+01 3.0e+00]] [1 1 1 ... 1 1 1]
smeared_fvt_training_ensemble_HH4b_800
[[5.0e-03 0.0e+00 0.0e+00 5.0e-01]
 [5.0e-03 0.0e+00 0.0e+00 1.0e+00]
 [5.0e-03 0.0e+00 0.0e+00 2.0e+00]
 ...
 [2.0e-02 4.9e+01 1.4e+01 1.0e+00]
 [2.0e-02 4.9e+01 1.4e+01 2.0e+00]
 [2.0e-02 4.9e+01 1.4e+01 3.0e+00]] [1 1 1 ... 1 1 1]


# STEP 3 Sanity Check

In [11]:
metadata = TrainingInfo.load_metadata()
for experiment_name in ["CR_fvt_training_ensemble_max", 
                        "CR_fvt_training_ensemble_max_HH4b_400", 
                        "CR_fvt_training_ensemble_max_HH4b_800"]:
    print(experiment_name)
    sanity_list = []
    hashes, hparams = TrainingInfo.find(
        {"experiment_name": experiment_name},
        return_hparams=True,
        use_cached_metadata=False
    )
    previous_step_experiment_name = "smeared_fvt_training_ensemble"
    if experiment_name == "CR_fvt_training_ensemble_max_HH4b_400":
        previous_step_experiment_name = "smeared_fvt_training_ensemble_HH4b_400"
    elif experiment_name == "CR_fvt_training_ensemble_max_HH4b_800":
        previous_step_experiment_name = "smeared_fvt_training_ensemble_HH4b_800"
        
    step_2_hashes, step_2_hparams = TrainingInfo.find(
        {
            "experiment_name": previous_step_experiment_name,
            "aux_info_step": 2,
        },
        return_hparams=True,
    )
    step_2_hashes_hparams_dict = {
        hash: hparam for hash, hparam in zip(step_2_hashes, step_2_hparams)
    }
    for hp in hparams:
        sr = hp["dataset"]["signal_ratio"]
        seed = hp["dataset"]["seed"]
        train_seed = hp["train_seed"]
        model_seed = hp["model_seed"]
        data_seed = hp["data_seed"]
        sr_size = hp["signal_region"]["4b_in_SR"]
        assert train_seed == model_seed == data_seed
        first_hash = hp["signal_region"]["SR_stats_hashes"][0]
        step_2_hparam = step_2_hashes_hparams_dict[first_hash]
        if hp["signal_region"]["stats_type"] == "smeared":
            noise_scale = step_2_hparam["smearing"]["noise_scale"]
        elif hp["signal_region"]["stats_type"] == "fvt":
            noise_scale = np.inf
        else:
            raise ValueError(f"Unknown stats_type: {hp['signal_region']['stats_type']}")

        sanity_list.append((sr, seed, train_seed, sr_size, noise_scale))
        
    unique_list, indices, counts = np.unique(sanity_list, return_counts=True, axis=0, 
                                    return_index=True)
    print(unique_list, counts)
    print(len(unique_list))
    print(np.sum(counts))
    assert np.all(counts == 1)

CR_fvt_training_ensemble_max
[[0.0e+00 0.0e+00 0.0e+00 5.0e-02 5.0e-01]
 [0.0e+00 0.0e+00 0.0e+00 5.0e-02 1.0e+00]
 [0.0e+00 0.0e+00 0.0e+00 5.0e-02 2.0e+00]
 ...
 [2.0e-02 4.9e+01 0.0e+00 2.0e-01 2.0e+00]
 [2.0e-02 4.9e+01 0.0e+00 2.0e-01 3.0e+00]
 [2.0e-02 4.9e+01 0.0e+00 2.0e-01     inf]] [1 1 1 ... 1 1 1]
5000
5000
CR_fvt_training_ensemble_max_HH4b_400
[[5.0e-03 0.0e+00 0.0e+00 5.0e-02 5.0e-01]
 [5.0e-03 0.0e+00 0.0e+00 5.0e-02 1.0e+00]
 [5.0e-03 0.0e+00 0.0e+00 5.0e-02 2.0e+00]
 ...
 [2.0e-02 4.9e+01 0.0e+00 2.0e-01 2.0e+00]
 [2.0e-02 4.9e+01 0.0e+00 2.0e-01 3.0e+00]
 [2.0e-02 4.9e+01 0.0e+00 2.0e-01     inf]] [1 1 1 ... 1 1 1]
4000
4000
CR_fvt_training_ensemble_max_HH4b_800
[[5.0e-03 0.0e+00 0.0e+00 5.0e-02 5.0e-01]
 [5.0e-03 0.0e+00 0.0e+00 5.0e-02 1.0e+00]
 [5.0e-03 0.0e+00 0.0e+00 5.0e-02 2.0e+00]
 ...
 [2.0e-02 4.9e+01 0.0e+00 2.0e-01 2.0e+00]
 [2.0e-02 4.9e+01 0.0e+00 2.0e-01 3.0e+00]
 [2.0e-02 4.9e+01 0.0e+00 2.0e-01     inf]] [1 1 1 ... 1 1 1]
4000
4000


In [9]:
# Sanity check

experiment_names
recent_experiment_names = [
    "CR_fvt_training_ensemble_max_fvt", 
    "CR_fvt_training_ensemble_max_fvt_HH4b_400",
    "CR_fvt_training_ensemble_max_smeared", 
    "CR_fvt_training_ensemble_max_smeared_HH4b_400",]

metadata = TrainingInfo.load_metadata()
existing_hashes = metadata.keys()
hashes_to_rerun = []
for experiment_name in recent_experiment_names:
    hashes, hparams = TrainingInfo.find(
        {"experiment_name": experiment_name},
        return_hparams=True,
        use_cached_metadata=True
    )
    for hash_, hparam in zip(hashes, hparams):
        SR_stat_hashes = hparam["signal_region"]["SR_stats_hashes"]
        non_existing_SR_stats_hashes_flag = False
        n_SR_stats_not_15_flag = False
        
        for SR_stat_hash in SR_stat_hashes:
            if SR_stat_hash not in existing_hashes:
                print(f"{hash_}: SR_stat_hash {SR_stat_hash} not in existing_hashes")
                print(experiment_name)
                dataset_hp = hparam["dataset"]
                print("signal_ratio", dataset_hp["signal_ratio"])
                # assert dataset_hp["signal_ratio"] == 0.02
                print("-"*100)
                non_existing_SR_stats_hashes_flag = True
        
        if len(SR_stat_hashes) != 15:
            print(f"{hash_}: has {len(SR_stat_hashes)} SR_stats")
            print(experiment_name)
            dataset_hp = hparam["dataset"]
            print("signal_ratio", dataset_hp["signal_ratio"])
            assert dataset_hp["signal_ratio"] in [0.0, 0.02]
            assert experiment_name in ["CR_fvt_training_ensemble_max_smeared",
                                        "CR_fvt_training_ensemble_max_fvt"]
            print("-"*100)
            n_SR_stats_not_15_flag = True
            
        if non_existing_SR_stats_hashes_flag or n_SR_stats_not_15_flag:
            assert dataset_hp["signal_ratio"] in [0.0, 0.02]
            # assert experiment_name in ["CR_fvt_training_ensemble_max_smeared",
            #                             "CR_fvt_training_ensemble_max_fvt"]
            hashes_to_rerun.append(hash_)


In [10]:
# Sanity check
recent_experiment_names = [
    "CR_fvt_training_ensemble_max_fvt", 
    "CR_fvt_training_ensemble_max_fvt_HH4b_400",
    "CR_fvt_training_ensemble_max_smeared", 
    "CR_fvt_training_ensemble_max_smeared_HH4b_400",
    ]

metadata = TrainingInfo.load_metadata()
existing_hashes = metadata.keys()
hashes_to_rerun = []
for experiment_name in recent_experiment_names:
    hashes, hparams = TrainingInfo.find(
        {"experiment_name": experiment_name},
        return_hparams=True,
    )
    for hash_, hparam in zip(hashes, hparams):
        SR_stat_hashes = hparam["signal_region"]["SR_stats_hashes"]
        non_existing_SR_stats_hashes_flag = False
        n_SR_stats_not_15_flag = False
        
        missing_SR_stat_hashes = []
        for SR_stat_hash in SR_stat_hashes:
            if SR_stat_hash not in existing_hashes:
                missing_SR_stat_hashes.append(SR_stat_hash)
        
        if len(missing_SR_stat_hashes) > 0:
            print(f"{hash_}: SR_stat_hash {missing_SR_stat_hashes} not in existing_hashes")
            print(experiment_name)
            dataset_hp = hparam["dataset"]
            print("signal_ratio", dataset_hp["signal_ratio"], 
                  "dataset_seed", dataset_hp["seed"])
            # assert dataset_hp["signal_ratio"] == 0.02
            print("-"*100)
            non_existing_SR_stats_hashes_flag = True
        
        if len(SR_stat_hashes) != 15:
            print(f"{hash_}: has {len(SR_stat_hashes)} SR_stats")
            print(experiment_name)
            dataset_hp = hparam["dataset"]
            print("signal_ratio", dataset_hp["signal_ratio"])
            assert dataset_hp["signal_ratio"] in [0.0, 0.02]
            assert experiment_name in ["CR_fvt_training_ensemble_max_smeared",
                                        "CR_fvt_training_ensemble_max_fvt"]
            print("-"*100)
            n_SR_stats_not_15_flag = True
            
        if non_existing_SR_stats_hashes_flag or n_SR_stats_not_15_flag:
            assert dataset_hp["signal_ratio"] in [0.0, 0.02]
            hashes_to_rerun.append(hash_)


In [11]:
for experiment_name in [
    "CR_fvt_training_ensemble_max_smeared", 
    "CR_fvt_training_ensemble_max_fvt",
    "CR_fvt_training_ensemble_max_smeared_HH4b_400", 
    "CR_fvt_training_ensemble_max_fvt_HH4b_400"
    ]:
    print(experiment_name)
    sanity_list = []
    hashes, hparams = TrainingInfo.find(
        {"experiment_name": experiment_name},
        return_hparams=True,
        use_cached_metadata=False
    )
    for hp in hparams:
        sr = hp["dataset"]["signal_ratio"]
        seed = hp["dataset"]["seed"]
        train_seed = hp["train_seed"]
        model_seed = hp["model_seed"]
        data_seed = hp["data_seed"]
        sr_size = hp["signal_region"]["4b_in_SR"]
        SR_stat_hashes = hp["signal_region"]["SR_stats_hashes"]
        noise_scale = metadata[SR_stat_hashes[0]]["smearing"]["noise_scale"]
        assert train_seed == model_seed == data_seed
        assert len(SR_stat_hashes) == 15
        sanity_list.append((sr, seed, train_seed, noise_scale, sr_size))

    unique_list, counts = np.unique(sanity_list, return_counts=True, axis=0)
    print(unique_list)
    assert np.all(counts == 1)
    if experiment_name == "CR_fvt_training_ensemble_max_smeared":
        if len(unique_list) != 5 * 50 * 4 * 6:
            print(f"Total number of unique configs is not correct, should be {5 * 50 * 4 * 6}")
            print(len(unique_list))
    elif experiment_name == "CR_fvt_training_ensemble_max_fvt":
        if len(unique_list) != 5 * 50 * 4:
            print(f"Total number of unique configs is not correct, should be {5 * 50 * 4}")
            print(len(unique_list))
    elif experiment_name == "CR_fvt_training_ensemble_max_fvt_HH4b_400":
        if len(unique_list) != 4 * 50 * 4:
            print(f"Total number of unique configs is not correct, should be {4 * 50 * 4}")
            print(len(unique_list))
    elif experiment_name == "CR_fvt_training_ensemble_max_smeared_HH4b_400":
        if len(unique_list) != 4 * 50 * 4 * 4:
            print(f"Total number of unique configs is not correct, should be {4 * 50 * 4 * 4}")
            print(len(unique_list))


CR_fvt_training_ensemble_max_smeared
[[0.0e+00 0.0e+00 0.0e+00 5.0e-01 5.0e-02]
 [0.0e+00 0.0e+00 0.0e+00 5.0e-01 1.0e-01]
 [0.0e+00 0.0e+00 0.0e+00 5.0e-01 1.5e-01]
 ...
 [2.0e-02 4.9e+01 0.0e+00 3.0e+00 1.0e-01]
 [2.0e-02 4.9e+01 0.0e+00 3.0e+00 1.5e-01]
 [2.0e-02 4.9e+01 0.0e+00 3.0e+00 2.0e-01]]
CR_fvt_training_ensemble_max_fvt
[[0.0e+00 0.0e+00 0.0e+00 1.0e+00 5.0e-02]
 [0.0e+00 0.0e+00 0.0e+00 1.0e+00 1.0e-01]
 [0.0e+00 0.0e+00 0.0e+00 1.0e+00 1.5e-01]
 ...
 [2.0e-02 4.9e+01 0.0e+00 1.0e+00 1.0e-01]
 [2.0e-02 4.9e+01 0.0e+00 1.0e+00 1.5e-01]
 [2.0e-02 4.9e+01 0.0e+00 1.0e+00 2.0e-01]]
CR_fvt_training_ensemble_max_smeared_HH4b_400
[[5.0e-03 0.0e+00 0.0e+00 5.0e-01 5.0e-02]
 [5.0e-03 0.0e+00 0.0e+00 5.0e-01 1.0e-01]
 [5.0e-03 0.0e+00 0.0e+00 5.0e-01 1.5e-01]
 ...
 [2.0e-02 4.9e+01 0.0e+00 3.0e+00 1.0e-01]
 [2.0e-02 4.9e+01 0.0e+00 3.0e+00 1.5e-01]
 [2.0e-02 4.9e+01 0.0e+00 3.0e+00 2.0e-01]]
CR_fvt_training_ensemble_max_fvt_HH4b_400
[[5.0e-03 0.0e+00 0.0e+00 1.0e+00 5.0e-02]
 [5.0e-

In [1]:
for experiment_name in [
    "CR_fvt_training_ensemble_max_smeared_HH4b_400", 
    "CR_fvt_training_ensemble_max_fvt_HH4b_400"
    ]:
    print(experiment_name)
    sanity_list = []
    hashes, hparams = TrainingInfo.find(
        {"experiment_name": experiment_name},
        return_hparams=True,
        use_cached_metadata=False
    )
    for hp in hparams:
        sr = hp["dataset"]["signal_ratio"]
        seed = hp["dataset"]["seed"]
        train_seed = hp["train_seed"]
        model_seed = hp["model_seed"]
        data_seed = hp["data_seed"]
        sr_size = hp["signal_region"]["4b_in_SR"]
        SR_stat_hashes = hp["signal_region"]["SR_stats_hashes"]
        noise_scale = metadata[SR_stat_hashes[0]]["smearing"]["noise_scale"]
        assert train_seed == model_seed == data_seed
        assert len(SR_stat_hashes) == 15
        sanity_list.append((sr, seed, train_seed, noise_scale, sr_size))

    unique_list, counts = np.unique(sanity_list, return_counts=True, axis=0)
    print(unique_list)
    assert np.all(counts == 1)
    if experiment_name == "CR_fvt_training_ensemble_max_smeared_HH4b_400":
        if len(unique_list) != 4 * 50 * 4 * 4:
            print(f"Total number of unique configs is not correct, should be {4 * 50 * 4 * 4}")
            print(len(unique_list))
    elif experiment_name == "CR_fvt_training_ensemble_max_fvt_HH4b_400":
        if len(unique_list) != 4 * 50 * 4:
            print(f"Total number of unique configs is not correct, should be {4 * 50 * 4}")
            print(len(unique_list))


CR_fvt_training_ensemble_max_smeared_HH4b_400


NameError: name 'TrainingInfo' is not defined

# Checking aux_info keys

In [95]:
import tqdm

recent_experiment_names = [
    "CR_fvt_training_ensemble_max_fvt", 
    "CR_fvt_training_ensemble_max_fvt_HH4b_400",
    "CR_fvt_training_ensemble_max_smeared", 
    "CR_fvt_training_ensemble_max_smeared_HH4b_400",
    "base_fvt_training_ensemble",
    "base_fvt_training_ensemble_HH4b_400",
    "base_fvt_training_ensemble_HH4b_800",
    "smeared_fvt_training_ensemble",
    "smeared_fvt_training_ensemble_HH4b_400",
]


TrainingInfo.load_cached_metadata.cache_clear()
for experiment_name in recent_experiment_names:
    print(experiment_name)
    hashes, hparams = TrainingInfo.find(
        {"experiment_name": experiment_name},
        return_hparams=True,
        use_cached_metadata=True
    )
    aux_info_keys = []
    random_idx = np.random.choice(range(len(hashes)), size=100, replace=False)
    for hash_ in tqdm.tqdm([hashes[i] for i in random_idx]):
        tinfo = TrainingInfo.load(hash_)
        aux_info_key = tuple(tinfo.aux_info.keys())
        if aux_info_key not in aux_info_keys:
            aux_info_keys.append(aux_info_key)
    print(aux_info_keys)

CR_fvt_training_ensemble_max_fvt


100%|██████████| 100/100 [00:03<00:00, 30.43it/s]


[('description', 'step', 'fvt_scores_train_SR', 'fvt_scores_tst_SR')]
CR_fvt_training_ensemble_max_fvt_HH4b_400


100%|██████████| 100/100 [00:03<00:00, 27.71it/s]


[('description', 'step', 'fvt_scores_train_SR', 'fvt_scores_tst_SR')]
CR_fvt_training_ensemble_max_smeared


...

[('description', 'step', 'base_fvt_logit_train', 'base_fvt_logit_tst'), ('description', 'step', 'base_fvt_score')]
smeared_fvt_training_ensemble


100%|██████████| 100/100 [00:00<00:00, 115.94it/s]


[('description', 'step', 'smeared_fvt_logit_train', 'smeared_fvt_logit_tst'), ('description', 'step')]
smeared_fvt_training_ensemble_HH4b_400


100%|██████████| 100/100 [00:01<00:00, 92.34it/s]

[('description', 'step', 'smeared_fvt_logit_train', 'smeared_fvt_logit_tst')]


In [107]:
import tqdm

recent_experiment_names = [
    "smeared_fvt_training_ensemble",
    "smeared_fvt_training_ensemble_HH4b_400",
]


TrainingInfo.load_cached_metadata.cache_clear()
for experiment_name in recent_experiment_names:
    print(experiment_name)
    hashes, hparams = TrainingInfo.find(
        {"experiment_name": experiment_name},
        return_hparams=True,
        use_cached_metadata=False
    )
    aux_info_keys = []
    random_idx = np.random.choice(range(len(hashes)), size=500, replace=False)
    for hash_ in tqdm.tqdm([hashes[i] for i in random_idx]):
        tinfo = TrainingInfo.load(hash_)
        aux_info_key = tuple(tinfo.aux_info.keys())
        if "smeared_fvt_logit_train" not in aux_info_key:
            print(f"{hash_} does not have smeared_fvt_logit_train")
            tinfo = TrainingInfo.load(hash_)
            print(f"hparams: {tinfo.hparams}")
        if aux_info_key not in aux_info_keys:
            aux_info_keys.append(aux_info_key)
    print(aux_info_keys)

smeared_fvt_training_ensemble


100%|██████████| 500/500 [00:04<00:00, 104.41it/s]


[('description', 'step', 'smeared_fvt_logit_train', 'smeared_fvt_logit_tst')]
smeared_fvt_training_ensemble_HH4b_400


100%|██████████| 500/500 [00:04<00:00, 110.44it/s]

[('description', 'step', 'smeared_fvt_logit_train', 'smeared_fvt_logit_tst')]


In [72]:
# tinfo = TrainingInfo.load("241228_171048_138204_IFaaCT")
# tinfo.hparams
# SR_stats_tst = tinfo.aux_info["SR_stats_tst"]
# SR_stats_train = tinfo.aux_info["SR_stats_train"]
# base_fvt_score_tst = tinfo.aux_info["base_fvt_score_tst"]
# base_fvt_score_train = tinfo.aux_info["base_fvt_score_train"]
# # smeared_fvt_logit_train = tinfo.aux_info["smeared_fvt_logit_train"]
# # smeared_fvt_logit_tst = tinfo.aux_info["smeared_fvt_logit_tst"]

# tmp_train =  np.log(base_fvt_score_train / (1 - base_fvt_score_train)) - SR_stats_train
# tmp_tst = np.log(base_fvt_score_tst / (1 - base_fvt_score_tst)) - SR_stats_tst
# # print(smeared_fvt_logit_tst)
# print(tmp_train)
# print(tmp_tst)
# # assert np.isclose(smeared_fvt_logit_tst, tmp_tst).all()
# # assert np.isclose(smeared_fvt_logit_tst, tmp_train).all()

In [74]:
from itertools import product

experiment_name = "smeared_fvt_training_ensemble"
previous_experiment_name = "base_fvt_training_ensemble"

TrainingInfo.load_cached_metadata.cache_clear()
for dataset_seed, signal_ratio in tqdm.tqdm(product(range(31, 50), [0.0, 0.005, 0.0075, 0.01, 0.02])):
    hashes, hparams = TrainingInfo.find(
        {
            "experiment_name": experiment_name,
            "dataset": lambda x: (
                x["seed"] == dataset_seed and x["signal_ratio"] == signal_ratio
            ),
        },
        use_cached_metadata=True,
        return_hparams=True,
    )

    tinfos = [TrainingInfo.load(h) for h in hashes]
    ms_hash = tinfos[0].ms_hash
    ms_idx = tinfos[0].ms_idx
    # signal_filename = tinfos[0].hparams["dataset"]["signal_filename"]
    # if any(tinfo.ms_hash != ms_hash for tinfo in tinfos):
    #     print(f"ms_hash mismatch for {hash_}")
    #     print("Dataset seed: ", dataset_seed, "Signal ratio: ", signal_ratio)
    # if any(np.any(tinfo.ms_idx != ms_idx) for tinfo in tinfos):
    #     print(f"ms_idx mismatch for {hash_}")
    #     print("Dataset seed: ", dataset_seed, "Signal ratio: ", signal_ratio)
    # if any(
    #     tinfo.hparams["dataset"]["signal_filename"] != signal_filename
    #     for tinfo in tinfos
    # ):
    #     print(f"signal_filename mismatch for {hash_}")
    #     print("Dataset seed: ", dataset_seed, "Signal ratio: ", signal_ratio)

    # train_seeds = np.unique([tinfo.hparams["train_seed"] for tinfo in tinfos])
    # if len(train_seeds) != 15:
    #     print(f"train_seeds mismatch for {hash_}")
    #     print("Dataset seed: ", dataset_seed, "Signal ratio: ", signal_ratio)
    encoder_hashes = np.unique([tinfo.hparams["encoder_hash"] for tinfo in tinfos])
    for encoder_hash in encoder_hashes:
        tinfo = TrainingInfo.load(encoder_hash)
        hp = tinfo.hparams
        # print(hp["dataset"]["seed"], hp["dataset"]["signal_ratio"], hp["train_seed"], hp["model_seed"], hp["data_seed"])
        
        if hp["experiment_name"] != previous_experiment_name:
            print(f"experiment_name mismatch for {encoder_hash}")
            print("Dataset seed: ", dataset_seed, "Signal ratio: ", signal_ratio)
    

95it [01:23,  1.14it/s]


In [24]:
# problematic hashes: 241017_161544_772029_onTwqd, 241017_161553_783800_Xg0Qi9

In [96]:
# from itertools import product

# experiment_name = "smeared_fvt_training_ensemble"
# dataset_seed = 31
# signal_ratio = 0.0

# hashes, hparams = TrainingInfo.find(
#     {
#         "experiment_name": experiment_name,
#         "dataset": lambda x: (
#             x["seed"] == dataset_seed and x["signal_ratio"] == signal_ratio
#         ),
#     },
#     use_cached_metadata=True,
#     return_hparams=True,
# )

# tinfos = [TrainingInfo.load(h) for h in hashes]
# corrupted_hashes = []
# for tinfo in tinfos:
#     if tinfo.hparams["encoder_hash"] == "241017_161544_772029_onTwqd":
#         print(tinfo.hparams["dataset"]["seed"], tinfo.hparams["dataset"]["signal_ratio"], tinfo.hparams["train_seed"], tinfo.hparams["model_seed"], tinfo.hparams["data_seed"])
#         print(tinfo.hash, tinfo.hparams["encoder_hash"])
#         print(tinfo.hparams)
#         corrupted_hashes.append(tinfo.hash)
        
    


In [97]:
tinfo = TrainingInfo.load("250312_130344_835046_JC0ifX")
print(tinfo.hparams)
tinfo.aux_info.keys()

{'data_seed': 14, 'dataloader': {'batch_size': 1024, 'batch_size_milestones': [1, 3, 6, 10, 15], 'batch_size_multiplier': 2}, 'depth': 8, 'early_stop_patience': None, 'encoder_mode': 'best', 'fit_batch_size': 1024, 'lr_scheduler': {'cooldown': 1, 'factor': 0.25, 'min_lr': 0.0002, 'patience': 3, 'threshold': 0.0001, 'type': 'ReduceLROnPlateau'}, 'max_epochs': 30, 'model': 'AttentionClassifier', 'model_seed': 14, 'optimizer': {'lr': 0.01, 'type': 'Adam'}, 'train_seed': 14, 'val_ratio': 0.33, 'experiment_name': 'smeared_fvt_training_ensemble', 'dataset': {'n_3b': 1000000, 'ratio_4b': 0.5, 'seed': 31, 'signal_filename': 'HH4b_picoAOD.h5', 'signal_ratio': 0.02}, 'smearing': {'hard_cutoff': False, 'noise_scale': 3.0, 'scale_mode': 'std', 'seed': 31}, 'step': 2, 'encoder_hash': '241223_172256_620849_EBi8iF', 'aux_info_description': 'Step 2: smeared_FvT_based_on_241223_172256_620849_EBi8iF', 'aux_info_step': 2}


dict_keys(['description', 'step'])